In [8]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.metrics import mean_squared_error
from tqdm import tqdm
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
# from pyearth import Earth  # MARS implementation
import xgboost as xgb
import catboost as cb
import warnings
warnings.filterwarnings("ignore")

In [5]:
# Read the training datasets
train1 = pd.read_csv('train.csv')
train2 = pd.read_csv('train_dataset.csv')

# Concatenate the two dataframes into one training dataset
train_data = pd.concat([train1, train2], ignore_index=True)
test = pd.read_csv('test.csv')

# Use all original features (all columns except 'id' and 'cost')
allfeatures = [col for col in train_data.columns if col not in ['id', 'cost']]
# List of features from the original dataset to be used in training
base_features = [
    'store_sqft', 
    'florist', 
    'salad_bar', 
    'prepared_food', 
    'coffee_bar', 
    'video_store', 
    'total_children', 
    'avg_cars_at home(approx).1', 
    'num_children_at_home'
]

# Define the features used to calculate the store_score
store_features = ['coffee_bar', 'video_store', 'salad_bar', 'prepared_food', 'florist']

# Set model parameters
params = {
    'objective': 'regression',
    'metric': 'rmse',
}

def get_categorical_features(features):
    return [col for col in features if col != 'store_score_ratio' and col != 'store_sales(in millions)' and col != 'unit_sales(in millions)' and col != 'gross_weight']

num_boost_round=100000
early_stopping_rounds=100


In [ ]:
# -------------------------------
# Phase 1: Model Experimentation
# -------------------------------
def model_comparison(train_data, features, random_state=42):
    # Load the unprocessed data
    # Build the training set: combine current train fold with all of train2
    X = train_data[features]
    y = np.log(train_data['cost'])

    # sample the data to reduce the size of the dataset for faster experimentation
    sample_size = 10000  # Or another manageable size
    indices = np.random.choice(X.shape[0], sample_size, replace=False)
    X_train = X.iloc[indices]
    y_train = y.iloc[indices]

    # If you have a specific target column, use:
    # X = data.drop('target_column_name', axis=1)
    # y = data['target_column_name']

    # Set up 5-fold cross-validation
    kf = KFold(n_splits=5, shuffle=True, random_state=random_state)

    # Define models with default parameters
    models = {
        'Support Vector Regression': SVR(),
        'K-Nearest Neighbors': KNeighborsRegressor(),
        'Random Forest': RandomForestRegressor(random_state=random_state),
        'Linear Regression': LinearRegression(),
        # 'MARS': Earth(),
        'LightGBM': lgb.LGBMRegressor(random_state=random_state),
        'XGBoost': xgb.XGBRegressor(random_state=random_state),
        'CatBoost': cb.CatBoostRegressor(random_state=random_state, verbose=0)
    }

    # Function to calculate RMSE
    def rmse_score(model, X, y):
        rmses = []
        for train_idx, val_idx in kf.split(X):
            X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
            y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]
            model.fit(X_train_fold, y_train_fold)
            preds = model.predict(X_val_fold)
            rmse = np.sqrt(mean_squared_error(y_val_fold, preds))
            rmses.append(rmse)
        return np.array(rmses)

    # Evaluate each model
    results = {}
    for name, model in models.items():
        print(f"\nEvaluating {name}...")
        cv_scores = rmse_score(model, X_train, y_train)
        results[name] = {
            'Mean CV RMSE': cv_scores.mean(),
            'Std CV RMSE': cv_scores.std()
        }
        print(f"{name}: Mean RMSE = {cv_scores.mean():.4f}, Std = {cv_scores.std():.4f}")

    # Sort models by performance (lower RMSE is better)
    sorted_models = sorted(results.items(), key=lambda x: x[1]['Mean CV RMSE'])
    print("\n--- Models Ranked by Performance ---")
    for i, (name, metrics) in enumerate(sorted_models):
        print(f"{i+1}. {name}: Mean RMSE = {metrics['Mean CV RMSE']:.4f}")

    # Train the best model on the full dataset (for demonstration)
    best_model_name = sorted_models[0][0]

    return best_model_name, models[best_model_name]

best_model_name, best_model = model_comparison(train1, allfeatures)
print(f"\nBest model: {best_model_name}")


Evaluating Support Vector Regression...
Support Vector Regression: Mean RMSE = 0.3202, Std = 0.0031

Evaluating K-Nearest Neighbors...
K-Nearest Neighbors: Mean RMSE = 0.3385, Std = 0.0036

Evaluating Random Forest...
Random Forest: Mean RMSE = 0.3169, Std = 0.0030

Evaluating Linear Regression...
Linear Regression: Mean RMSE = 0.3171, Std = 0.0021

Evaluating LightGBM...
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000393 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 571
[LightGBM] [Info] Number of data points in the train set: 8000, number of used features: 15
[LightGBM] [Info] Start training from score 4.550897
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, th

In [15]:
# -------------------------------
# Cross-Validation Function Using train1 for Folds and train2 Always for Training
# -------------------------------

def run_cv_with_train2(train1, train2, 
                       num_boost_round=num_boost_round, 
                       early_stopping_rounds=early_stopping_rounds, 
                       random_state=5):
    """
    Performs 5-fold CV on train1, but for each fold, the training data is the union of:
      - The training fold from train1
      - All of train2 (which is never used for validation)
      
    The target is log-transformed (np.log(cost)) and RMSE is computed on the log scale.
    
    Parameters:
      train1: DataFrame from train.csv (used for fold splitting and validation)
      train2: DataFrame from train_dataset.csv (always used for training)
      num_boost_round: Maximum number of boosting iterations.
      early_stopping_rounds: Early stopping rounds.
      random_state: Seed for reproducibility.
      
    Returns:
      Average RMSE (on the log scale) across the 5 folds.
    """
    kf = KFold(n_splits=5, shuffle=True, random_state=random_state)
    rmses = []
    
    # Loop over folds using train1 only
    for train_idx, val_idx in kf.split(train1):
        # Get the training fold from train1 and the validation fold from train1
        train1_fold = train1.iloc[train_idx]
        val_fold = train1.iloc[val_idx]
        
        # Build the training set: combine current train1 fold with all of train2
        X_train = pd.concat([train1_fold, train2], ignore_index=True)[allfeatures]
        y_train = pd.concat([np.log(train1_fold['cost']), np.log(train2['cost'])], ignore_index=True)
        
        # Validation is done only on the hold-out fold from train1
        X_val = val_fold[allfeatures]
        y_val = np.log(val_fold['cost'])
        
        training_data = lgb.Dataset(X_train, label=y_train)
        validation_data = lgb.Dataset(X_val, label=y_val, free_raw_data=False)
        
        model = lgb.train(
            params,
            training_data,
            num_boost_round=num_boost_round,
            valid_sets=[validation_data],
            callbacks=[lgb.early_stopping(early_stopping_rounds), lgb.log_evaluation(100)]
        )
        
        preds = model.predict(X_val, num_iteration=model.best_iteration)
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        rmses.append(rmse)
        
    return np.mean(rmses)

# -------------------------------
# Evaluate CV Errors
# -------------------------------

# CV error when using only train1 (both for training and validation)
def run_cv_train1_only(data, num_boost_round=num_boost_round, early_stopping_rounds=early_stopping_rounds, random_state=5):
    kf = KFold(n_splits=5, shuffle=True, random_state=random_state)
    rmses = []
    for train_idx, val_idx in kf.split(data):
        X_train = data.iloc[train_idx][allfeatures]
        X_val = data.iloc[val_idx][allfeatures]
        y_train = np.log(data.iloc[train_idx]['cost'])
        y_val = np.log(data.iloc[val_idx]['cost'])
        
        training_data = lgb.Dataset(X_train, label=y_train)
        validation_data = lgb.Dataset(X_val, label=y_val, free_raw_data=False)
        
        model = lgb.train(
            params,
            training_data,
            num_boost_round=num_boost_round,
            valid_sets=[validation_data],
            callbacks=[lgb.early_stopping(early_stopping_rounds), lgb.log_evaluation(100)]
        )
        
        preds = model.predict(X_val, num_iteration=model.best_iteration)
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        rmses.append(rmse)
    return np.mean(rmses)

cv_error_train1 = run_cv_train1_only(train1)
cv_error_combined = run_cv_with_train2(train1, train2)

print("Average 5-fold CV RMSE (log scale) using train.csv only:", cv_error_train1)
print("Average 5-fold CV RMSE (log scale) using train.csv + train_dataset.csv:", cv_error_combined)


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003063 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 579
[LightGBM] [Info] Number of data points in the train set: 288268, number of used features: 15
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Start training from score 4.552538
Training until validation scores don't improve for 100 rounds
[100]	valid_0's rmse: 0.302853
[200]	valid_0's rmse: 0.301618
[300]	valid_0's rmse: 0.301212
[400]	valid_0's rmse: 0.300916
[500]	valid_0's rmse: 0.300681
[600]	valid_0's rmse: 0.300465
Early stopping, best iteration is:
[597]	valid_0's rmse: 0.300464
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise

In [ ]:
# Create additional features for train dataset
train_data['store_score'] = train_data[store_features].sum(axis=1)
train_data['store_score_ratio'] = train_data['store_sqft'] / train_data['store_score']

# Create additional features for test dataset
test['store_score'] = test[store_features].sum(axis=1)
test['store_score_ratio'] = test['store_sqft'] / test['store_score']

In [ ]:
# 3. Base features + store_score
base_plus_store_score = base_features + ['store_score']

# 4. Base features + store_score + store_score_ratio
base_plus_store_score_ratio = base_features + ['store_score', 'store_score_ratio']

# -------------------------------
# 5-Fold CV Function
# -------------------------------
def run_cv_features(features_list, num_boost_round=num_boost_round, early_stopping_rounds=early_stopping_rounds, random_state=42):
    """
    Performs 5-fold CV using LightGBM on the combined training data.
    The target is log-transformed (np.log(cost)). RMSE is computed on the log scale.
    
    Parameters:
      features_list: List of column names to use for training.
      num_boost_round: Maximum number of boosting iterations.
      early_stopping_rounds: Early stopping rounds.
      random_state: Seed for reproducibility.
      
    Returns:
      Average RMSE across folds (on the log scale).
    """
    kf = KFold(n_splits=5, shuffle=True, random_state=random_state)
    rmses = []
    
    for train_idx, val_idx in kf.split(train_data):
        X_train = train_data.iloc[train_idx][features_list]
        X_val = train_data.iloc[val_idx][features_list]
        # Log-transform the target
        y_train = np.log(train_data.iloc[train_idx]['cost'])
        y_val = np.log(train_data.iloc[val_idx]['cost'])
        
        training_data = lgb.Dataset(X_train, label=y_train)
        validation_data = lgb.Dataset(X_val, label=y_val, free_raw_data=False)
        
        model = lgb.train(
            params,
            training_data,
            num_boost_round=num_boost_round,
            valid_sets=[validation_data],
            callbacks=[lgb.early_stopping(early_stopping_rounds), lgb.log_evaluation(100)]
        )
        
        preds = model.predict(X_val, num_iteration=model.best_iteration)
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        rmses.append(rmse)
        
    return np.mean(rmses)

# -------------------------------
# Evaluate Each Feature Set
# -------------------------------
cv_rmse_all_features = run_cv_features(allfeatures)
cv_rmse_base_features = run_cv_features(base_features)
cv_rmse_base_plus_store_score = run_cv_features(base_plus_store_score)
cv_rmse_base_plus_store_score_ratio = run_cv_features(base_plus_store_score_ratio)

print("Average 5-fold CV RMSE (log scale) using all features:", cv_rmse_all_features)
print("Average 5-fold CV RMSE (log scale) using base features:", cv_rmse_base_features)
print("Average 5-fold CV RMSE (log scale) using base features + store_score:", cv_rmse_base_plus_store_score)
print("Average 5-fold CV RMSE (log scale) using base features + store_score + store_score_ratio:", cv_rmse_base_plus_store_score_ratio)


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003282 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 578
[LightGBM] [Info] Number of data points in the train set: 329359, number of used features: 15
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Start training from score 4.551981
Training until validation scores don't improve for 100 rounds
[100]	valid_0's rmse: 0.303606
[200]	valid_0's rmse: 0.301522
[300]	valid_0's rmse: 0.300947
[400]	valid_0's rmse: 0.3008
[500]	valid_0's rmse: 0.300786
Early stopping, best iteration is:
[488]	valid_0's rmse: 0.300775
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of

In [25]:
optimal_features = base_features + ['store_score', 'store_score_ratio']

In [ ]:
# -------------------------------
# Cross-Validation Function
# -------------------------------
def run_cv_optimal(data, use_categorical=False, num_boost_round=num_boost_round, early_stopping_rounds=early_stopping_rounds, random_state=42):
    """
    Performs 5-fold CV using LightGBM on the combined training data (data).
    The target is log-transformed (np.log(cost)), and RMSE is computed on the log scale.
    
    Parameters:
      data: DataFrame containing the combined training data (must include 'cost').
      use_categorical: If True, specify categorical features using get_categorical_features.
      num_boost_round: Maximum number of boosting iterations.
      early_stopping_rounds: Early stopping rounds.
      random_state: Seed for reproducibility.
      
    Returns:
      Average RMSE (on the log scale) across folds.
    """
    kf = KFold(n_splits=5, shuffle=True, random_state=random_state)
    rmses = []
    
    # Determine categorical features if needed.
    if use_categorical:
        cat_features = get_categorical_features(optimal_features)
    else:
        cat_features = None  # Do not pass categorical_feature parameter
    
    for train_idx, val_idx in kf.split(data):
        X_train = data.iloc[train_idx][optimal_features]
        X_val = data.iloc[val_idx][optimal_features]
        
        # Log-transform the target.
        y_train = np.log(data.iloc[train_idx]['cost'])
        y_val = np.log(data.iloc[val_idx]['cost'])
        
        if use_categorical:
            training_data = lgb.Dataset(X_train, label=y_train, categorical_feature=cat_features)
            validation_data = lgb.Dataset(X_val, label=y_val, categorical_feature=cat_features, free_raw_data=False)
        else:
            training_data = lgb.Dataset(X_train, label=y_train)
            validation_data = lgb.Dataset(X_val, label=y_val, free_raw_data=False)
        
        model = lgb.train(
            params,
            training_data,
            num_boost_round=num_boost_round,
            valid_sets=[validation_data],
            callbacks=[lgb.early_stopping(early_stopping_rounds), lgb.log_evaluation(100)]
        )
        
        preds = model.predict(X_val, num_iteration=model.best_iteration)
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        rmses.append(rmse)
        
    return np.mean(rmses)

# -------------------------------
# Evaluate the Two Approaches
# -------------------------------

# Assuming your combined training data is in 'train'
cv_rmse_numerical = run_cv_optimal(train_data, use_categorical=False)
cv_rmse_categorical = run_cv_optimal(train_data, use_categorical=True)

print("Average 5-fold CV RMSE (log scale) treating all features as numerical:", cv_rmse_numerical)
print("Average 5-fold CV RMSE (log scale) with categorical variables specified:", cv_rmse_categorical)


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003065 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 106
[LightGBM] [Info] Number of data points in the train set: 329359, number of used features: 11
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Start training from score 4.551981
Training until validation scores don't improve for 100 rounds
[100]	valid_0's rmse: 0.302293
[200]	valid_0's rmse: 0.299502
[300]	valid_0's rmse: 0.29837
[400]	valid_0's rmse: 0.297714
[500]	valid_0's rmse: 0.297425
[600]	valid_0's rmse: 0.29721
[700]	valid_0's rmse: 0.29711
[800]	valid_0's rmse: 0.297041
[900]	valid_0's rmse: 0.297004
[1000]	valid_0's rmse: 0.296985
[1100]	valid_0's rmse: 0.296977
[1200]	valid_0's rmse: 0.296974


In [64]:
params = {
    'objective': 'regression',
    'metric': 'rmse',
    "verbosity": -1,
    "verbose_eval": False,
}

def train_lgb_model_random_split(data, categorical_features=get_categorical_features(optimal_features), params=params, num_boost_round=num_boost_round, early_stopping_rounds=early_stopping_rounds, val_size=0.2):
    """
    Create a random train-validation split from the full training data,
    then train a LightGBM model using early stopping on the validation set.
    """
    X = data[optimal_features]
    y = np.log(data['cost'])
    # Each split will be different; random_state can be set to None for true randomness.
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=val_size, shuffle=True)
    train_dataset = lgb.Dataset(X_train, label=y_train, categorical_feature=categorical_features)
    val_dataset = lgb.Dataset(X_val, label=y_val, categorical_feature=categorical_features, free_raw_data=False)
    
    model = lgb.train(
        params,
        train_dataset,
        num_boost_round=num_boost_round,
        valid_sets=[val_dataset],
        callbacks=[lgb.early_stopping(early_stopping_rounds, verbose=False)],
    )
    
    return model

def bagged_models_random_split(data, categorical_features=get_categorical_features(optimal_features), params=params, num_boost_round=num_boost_round, early_stopping_rounds=early_stopping_rounds, num_models=100, val_size=0.2):
    """
    Train multiple LightGBM models using different random train-validation splits.
    Returns a list of trained models.
    """
    model_list = []
    dataiter = tqdm(range(num_models), desc="Training", leave=True, dynamic_ncols=True)
    for i in dataiter:
        model = train_lgb_model_random_split(data, categorical_features, 
                                             params, num_boost_round, early_stopping_rounds, val_size)
        model_list.append(model)
        #print(f"Trained model {i+1}/{num_models}")
    return model_list

def predict_bagged_models(test_data, model_list, features=optimal_features):
    """
    Average predictions from a list of LightGBM models.
    """
    preds = np.zeros(test_data.shape[0])
    for model in model_list:
        preds += model.predict(test_data[features], num_iteration=model.best_iteration)
    return preds / len(model_list)

In [ ]:
# List of candidate bag sizes.
bag_sizes = [1, 25, 50, 100, 200]

for bs in bag_sizes:
    print(f"\nTraining bagged models with bag size: {bs}")
    # Train an ensemble (bag) with the specified number of models.
    models = bagged_models_random_split(train_data, 
                                          categorical_features=get_categorical_features(optimal_features), 
                                          params=params, 
                                          num_boost_round=num_boost_round, 
                                          early_stopping_rounds=early_stopping_rounds, 
                                          num_models=bs, 
                                          val_size=0.2)
    
    # Predict on the test set using the bagged ensemble.
    preds_log = predict_bagged_models(test, models, features=optimal_features)
    
    # Convert predictions back to original cost scale.
    preds = np.exp(preds_log)
    
    # Create submission DataFrame.
    submission = pd.DataFrame({
        'id': test['id'],
        'cost': preds
    })
    
    # Save submission CSV file (e.g., submission_bag_25.csv).
    submission_filename = f"submission_bag_{bs}.csv"
    submission.to_csv(submission_filename, index=False)
    print(f"Saved submission for bag size {bs} to {submission_filename}")



Training bagged models with bag size: 1



Training: 100%|███████████████████████████████████| 1/1 [00:09<00:00,  9.62s/it]


Saved submission for bag size 1 to submission_bag_1.csv

Training bagged models with bag size: 25



ining:  32%|██████████▉                       | 8/25 [01:19<02:50, 10.05s/it]

[LightGBM] [Warning] Met categorical feature which contains sparse values. Consider renumbering to consecutive integers started from zero



Training: 100%|█████████████████████████████████| 25/25 [04:08<00:00,  9.94s/it]


... (49 outputs left out)

Training:  95%|█████████████████████████████▍ | 190/200 [32:19<01:37,  9.71s/it]

[LightGBM] [Warning] Met categorical feature which contains sparse values. Consider renumbering to consecutive integers started from zero


Training: 100%|███████████████████████████████| 200/200 [33:59<00:00, 10.20s/it]


Saved submission for bag size 200 to submission_bag_200.csv
